In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Brain Age Gap (BAG) Pipeline
=============================

Workflow:
  1. Data preprocessing: read cleaned CSVs for 3 modalities, encode sex,
     merge disease labels, split healthy subjects into train/test,
     keep diseased subjects separately.
  2. Train mode: for 3 modalities x 2 sexes = 6 subsets, train LinearSVR
     with 20-fold CV, apply age-bias correction, retrain on full data,
     save as pkl.
  3. Predict mode: load pkl files, compute BAG (raw gap and bias-corrected
     gap_resid) for healthy test set and diseased subjects.
  4. Both mode: train then predict.

Outputs:
  - Per-modality pkl files (compatible with existing notebooks)
  - Per-modality CSV (eid, age, age_predic, gap, gap_resid, sex, group)
  - Combined long CSV (all subjects x 3 modalities)
  - Combined wide CSV (one row per eid, BAG columns for each modality)

Parallelization (designed for ~100-core server):
  - 6 outer tasks (3 modality x 2 sex): N_JOBS_OUTER
  - K-fold CV within each task: N_JOBS_INNER
  - Total processes ~ N_JOBS_OUTER x N_JOBS_INNER
"""

import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from joblib import Parallel, delayed

from sklearn.svm import LinearSVR
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr

warnings.filterwarnings("ignore")


# =============================================================================
# Configuration
# =============================================================================

# --- Run mode ---
# "train"   : train only (saves pkl files)
# "predict" : predict only (requires existing pkl files)
# "both"    : train + predict
MODE = "predict"

# --- Input: cleaned CSV paths for 3 modalities ---
CLEANED_FILES = {
    "brain":  "./data/disease/brain_cleaned.csv",
    "huizhi": "./data/disease/huizhi_cleaned.csv",
    "baizhi": "./data/disease/baizhi_cleaned.csv",
}

# --- Disease label file ---
DISEASE_FILE = "./data/disease_condition_0113.csv"

# --- Output directory ---
OUTPUT_DIR = "./results/brain_age_output"

# --- Random seed ---
RANDOM_SEED = 123

# --- Pre-existing train/test splits (from R code) ---
# Set to {} to let this script create new splits (80/20, seed=123)
# When provided, these CSV paths are used directly for exact reproducibility
EXISTING_SPLIT_CSVS = {
    "brain":  {"train": "./data/brain_age/train_brain_all1213.csv",
               "test":  "./data/brain_age/test_brain_all1213.csv"},
    "huizhi": {"train": "./data/brain_age/train_huizhi_all1213.csv",
               "test":  "./data/brain_age/test_huizhi_all1213.csv"},
    "baizhi": {"train": "./data/brain_age/train_baizhi_all1213.csv",
               "test":  "./data/brain_age/test_baizhi_all1213.csv"},
}

# --- Training hyperparameters ---
K_FOLDS = 20
SVR_C = 1.0
SVR_EPSILON = 0.1
SVR_MAX_ITER = 10000

# --- Parallelization ---
N_JOBS_OUTER = 2      # Number of (modality, sex) tasks to run in parallel
N_JOBS_INNER = 20     # K-fold parallelism within each task

# --- Sex encoding (Female=0, Male=1) ---
SEX_VALUES = [0, 1]

# --- Modalities ---
MODALITIES = ["brain", "huizhi", "baizhi"]


# =============================================================================
# Utility Functions
# =============================================================================

def log(msg):
    """Print a timestamped log message."""
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def ensure_dir(path):
    """Create directory if it does not exist."""
    Path(path).mkdir(parents=True, exist_ok=True)


def extract_feature_columns(df):
    """Extract feature column names by excluding metadata columns."""
    drop_cols = {"age", "sex", "eid", "Unnamed: 0", "X", "disease"}
    return [c for c in df.columns if c not in drop_cols]


# =============================================================================
# Step 1: Data Preprocessing
# =============================================================================

def preprocess_one_modality(modality, cleaned_path, disease_df, out_dir, seed=123):
    """Preprocess one modality: clean, merge disease labels, split train/test.

    Outputs:
      - train_{modality}.csv  (healthy 80%)
      - test_{modality}.csv   (healthy 20%)
      - diseased_{modality}.csv

    Returns (train_df, test_df, diseased_df).
    """
    log(f"[preprocess/{modality}] Reading {cleaned_path}")
    df = pd.read_csv(cleaned_path)

    # Drop redundant index columns
    for col in ["X", "Unnamed: 0"]:
        if col in df.columns:
            df = df.drop(columns=[col])

    # Rename columns
    df = df.rename(columns={"p31": "sex", "p21022": "age"})

    # Encode sex: Female=0, Male=1
    df["sex"] = pd.to_numeric(
        df["sex"].replace({"Female": 0, "Male": 1}), errors="coerce"
    )

    # Drop rows with any NaN
    df = df.dropna().reset_index(drop=True)
    log(f"[preprocess/{modality}] After dropna: n={len(df)}")

    # Merge disease labels
    merged = df.merge(disease_df, on="eid", how="left")
    healthy = merged[merged["disease"] == 0].drop(columns=["disease"]).reset_index(drop=True)
    diseased = merged[merged["disease"] == 1].drop(columns=["disease"]).reset_index(drop=True)
    log(f"[preprocess/{modality}] Healthy n={len(healthy)}, diseased n={len(diseased)}")

    # 80/20 train/test split on healthy subjects
    rng = np.random.RandomState(seed)
    n = len(healthy)
    train_size = int(np.floor(0.8 * n))
    idx = rng.permutation(n)
    train_df = healthy.iloc[idx[:train_size]].reset_index(drop=True)
    test_df = healthy.iloc[idx[train_size:]].reset_index(drop=True)
    log(f"[preprocess/{modality}] Train n={len(train_df)}, test n={len(test_df)}")

    # Save
    ensure_dir(out_dir)
    train_df.to_csv(Path(out_dir) / f"train_{modality}.csv", index=False)
    test_df.to_csv(Path(out_dir) / f"test_{modality}.csv", index=False)
    diseased.to_csv(Path(out_dir) / f"diseased_{modality}.csv", index=False)

    return train_df, test_df, diseased


def load_existing_split(modality, paths):
    """Load pre-existing train/test split CSVs (e.g., from R code)."""
    train_df = pd.read_csv(paths["train"])
    test_df = pd.read_csv(paths["test"])
    for col in ["Unnamed: 0", "X"]:
        for d in (train_df, test_df):
            if col in d.columns:
                d.drop(columns=[col], inplace=True)
    return train_df, test_df


# =============================================================================
# Step 2: Training
# =============================================================================

def train_one_fold(k, train_idx, test_idx, X, y):
    """Train and predict for a single CV fold (called by joblib.Parallel)."""
    xTr, xTe = X[train_idx], X[test_idx]
    yTr = y[train_idx]
    scaler = StandardScaler()
    xTr_s = scaler.fit_transform(xTr)
    xTe_s = scaler.transform(xTe)
    model = LinearSVR(C=SVR_C, epsilon=SVR_EPSILON, max_iter=SVR_MAX_ITER)
    model.fit(xTr_s, yTr)
    yhat = model.predict(xTe_s)
    return test_idx, yhat


def train_one_subset(modality, gender, train_df, out_dir, n_jobs_inner):
    """Full training pipeline for one (modality, gender) subset.

    Steps:
      1. Filter by sex
      2. K-fold CV to get out-of-fold predictions
      3. Fit age-bias correction (linear regression of gap on age)
      4. Retrain final model on all data
      5. Save pkl
    """
    feat_cols = extract_feature_columns(train_df)
    sub = train_df[train_df["sex"] == gender].reset_index(drop=True)
    X = sub[feat_cols].values.astype(np.float64)
    y = sub["age"].values.astype(np.float64)
    n = len(y)
    log(f"[train/{modality}/sex{gender}] n={n}, n_features={X.shape[1]}")

    # K-fold CV
    kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
    age_predic = np.zeros(n)

    t0 = time.time()
    results = Parallel(n_jobs=n_jobs_inner, prefer="processes")(
        delayed(train_one_fold)(k, tr, te, X, y)
        for k, (tr, te) in enumerate(kf.split(X))
    )
    for test_idx, yhat in results:
        age_predic[test_idx] = yhat
    log(f"[train/{modality}/sex{gender}] K-fold done in {time.time()-t0:.1f}s")

    # OOF performance
    r_predic, _ = pearsonr(y, age_predic)
    mae = mean_absolute_error(y, age_predic)
    log(f"[train/{modality}/sex{gender}] OOF r={r_predic:.3f}, MAE={mae:.3f}")

    # Age-bias correction: regress gap on true age
    gap = age_predic - y
    beta_model = LinearRegression()
    beta_model.fit(y.reshape(-1, 1), gap)

    # Retrain final model on full training data
    scaler_final = StandardScaler()
    X_s = scaler_final.fit_transform(X)
    final_model = LinearSVR(C=SVR_C, epsilon=SVR_EPSILON, max_iter=SVR_MAX_ITER)
    final_model.fit(X_s, y)

    # Save pkl
    save_path = Path(out_dir) / f"model_{modality}_sex{gender}.pkl"
    joblib.dump({
        "model": final_model,
        "scaler": scaler_final,
        "age_real": y,
        "age_predic": age_predic,
        "r_predic": r_predic,
        "mae": mae,
        "sex": gender,
        "modality": modality,
        "beta_model": beta_model,
        "feat_cols": feat_cols,
    }, save_path)
    log(f"[train/{modality}/sex{gender}] Model saved to {save_path}")

    return save_path


def run_train_tasks(train_dfs, out_dir):
    """Run all 6 training tasks (3 modalities x 2 sexes) in parallel."""
    tasks = [(m, g) for m in MODALITIES for g in SEX_VALUES]
    log(f"Training tasks: {len(tasks)} (modality x sex), "
        f"outer_jobs={N_JOBS_OUTER}, inner_jobs={N_JOBS_INNER}")

    def _run(modality, gender):
        return train_one_subset(modality, gender, train_dfs[modality],
                                out_dir, N_JOBS_INNER)

    Parallel(n_jobs=N_JOBS_OUTER, prefer="processes")(
        delayed(_run)(m, g) for (m, g) in tasks
    )


# =============================================================================
# Step 3: Prediction
# =============================================================================

def predict_with_model(model_pkl_path, df, group_label):
    """Predict brain age using a saved model pkl.

    Returns a DataFrame with columns:
      eid, sex, age, age_predic, gap, gap_resid, modality, group
    """
    md = joblib.load(model_pkl_path)
    model = md["model"]
    scaler = md["scaler"]
    beta_model = md["beta_model"]
    gender = md["sex"]
    modality = md.get("modality", "unknown")
    feat_cols = md.get("feat_cols", None)

    # Filter by sex
    sub = df[df["sex"] == gender].reset_index(drop=True)
    if len(sub) == 0:
        return pd.DataFrame(columns=["eid", "sex", "age", "age_predic",
                                     "gap", "gap_resid", "modality", "group"])

    # Select feature columns
    if feat_cols is None:
        feat_cols = extract_feature_columns(sub)
    missing = [c for c in feat_cols if c not in sub.columns]
    if missing:
        raise ValueError(f"[predict/{modality}/sex{gender}/{group_label}] "
                         f"Missing feature columns: {missing[:5]}... ({len(missing)} total)")

    X = sub[feat_cols].values.astype(np.float64)
    y = sub["age"].values.astype(np.float64)

    # Predict
    X_s = scaler.transform(X)
    age_pred = model.predict(X_s)

    # Compute gap and bias-corrected gap_resid
    gap = age_pred - y
    coef = float(np.ravel(beta_model.coef_)[0])
    intercept = float(np.ravel(beta_model.intercept_)[0])
    age_fit = coef * y + intercept
    gap_resid = gap - age_fit

    out = pd.DataFrame({
        "eid": sub["eid"].values,
        "sex": sub["sex"].values,
        "age": y,
        "age_predic": age_pred,
        "gap": gap,
        "gap_resid": gap_resid,
        "modality": modality,
        "group": group_label,
    })

    if len(out) >= 2:
        r_val, _ = pearsonr(y, age_pred)
        mae_val = mean_absolute_error(y, age_pred)
        log(f"[predict/{modality}/sex{gender}/{group_label}] n={len(out)}, "
            f"r={r_val:.3f}, MAE={mae_val:.3f}")
    return out


def build_train_oof_bag(model_pkl_path, train_df):
    """Build BAG for training set using stored out-of-fold predictions.

    Uses the OOF predictions saved in the pkl (no re-prediction needed),
    ensuring the same evaluation paradigm as test/diseased sets.
    """
    md = joblib.load(model_pkl_path)
    gender = md["sex"]
    modality = md.get("modality", "unknown")
    age_real = md["age_real"]
    age_oof = md["age_predic"]
    beta_model = md["beta_model"]

    # Get the matching subset (same order as during training)
    sub = train_df[train_df["sex"] == gender].reset_index(drop=True)

    # Alignment check
    if len(sub) != len(age_real):
        raise ValueError(
            f"[train_oof/{modality}/sex{gender}] "
            f"Subset size {len(sub)} != pkl age_real size {len(age_real)}"
        )
    if not np.allclose(sub["age"].values.astype(float), age_real, atol=1e-6):
        raise ValueError(
            f"[train_oof/{modality}/sex{gender}] "
            f"Age values do not match pkl -- has the training CSV been modified?"
        )

    # Compute gap and bias-corrected gap_resid
    gap = age_oof - age_real
    coef = float(np.ravel(beta_model.coef_)[0])
    intercept = float(np.ravel(beta_model.intercept_)[0])
    age_fit = coef * age_real + intercept
    gap_resid = gap - age_fit

    out = pd.DataFrame({
        "eid": sub["eid"].values,
        "sex": sub["sex"].values,
        "age": age_real,
        "age_predic": age_oof,
        "gap": gap,
        "gap_resid": gap_resid,
        "modality": modality,
        "group": "train_healthy_oof",
    })

    r_val, _ = pearsonr(age_real, age_oof)
    mae_val = mean_absolute_error(age_real, age_oof)
    log(f"[train_oof/{modality}/sex{gender}] n={len(out)}, "
        f"r={r_val:.3f}, MAE={mae_val:.3f}")
    return out


def run_predict_tasks(test_dfs, diseased_dfs, out_dir, train_dfs=None):
    """Run prediction for all modalities and output CSV files.

    Outputs:
      - Per-modality CSV: bag_{modality}.csv
      - Long-format CSV: bag_all_long.csv (all subjects x all modalities)
      - Wide-format CSV: bag_all_wide.csv (one row per eid, BAG for each modality)
    """
    all_rows = []
    per_modality_rows = {m: [] for m in MODALITIES}

    for modality in MODALITIES:
        for gender in SEX_VALUES:
            pkl_path = Path(out_dir) / f"model_{modality}_sex{gender}.pkl"
            if not pkl_path.exists():
                log(f"Warning: model file not found, skipping: {pkl_path}")
                continue

            # Training set BAG (from stored OOF predictions)
            if train_dfs is not None and modality in train_dfs and train_dfs[modality] is not None:
                r = build_train_oof_bag(pkl_path, train_dfs[modality])
                per_modality_rows[modality].append(r)
                all_rows.append(r)

            # Healthy test set
            if modality in test_dfs and test_dfs[modality] is not None:
                r = predict_with_model(pkl_path, test_dfs[modality], group_label="test_healthy")
                per_modality_rows[modality].append(r)
                all_rows.append(r)

            # Diseased subjects
            if modality in diseased_dfs and diseased_dfs[modality] is not None:
                r = predict_with_model(pkl_path, diseased_dfs[modality], group_label="diseased")
                per_modality_rows[modality].append(r)
                all_rows.append(r)

    # --- Per-modality CSV ---
    ensure_dir(out_dir)
    for modality, rows in per_modality_rows.items():
        if not rows:
            continue
        df_m = pd.concat(rows, axis=0, ignore_index=True)
        out_csv = Path(out_dir) / f"bag_{modality}.csv"
        df_m.to_csv(out_csv, index=False)
        log(f"[output] {modality} BAG saved to {out_csv} (n={len(df_m)})")

    # --- Long-format CSV (all modalities) ---
    if all_rows:
        df_long = pd.concat(all_rows, axis=0, ignore_index=True)
        long_csv = Path(out_dir) / "bag_all_long.csv"
        df_long.to_csv(long_csv, index=False)
        log(f"[output] Long table saved to {long_csv} (n={len(df_long)})")

        # --- Wide-format CSV (one row per eid) ---
        wide_vals = df_long.pivot_table(
            index="eid",
            columns="modality",
            values=["age_predic", "gap", "gap_resid"],
            aggfunc="first",
        )
        wide_vals.columns = [f"{metric}_{mod}" for metric, mod in wide_vals.columns]
        wide_vals = wide_vals.reset_index()

        # Attach sex and age (take first occurrence per eid)
        meta = df_long.groupby("eid")[["sex", "age"]].first().reset_index()
        wide = meta.merge(wide_vals, on="eid", how="left")

        wide_csv = Path(out_dir) / "bag_all_wide.csv"
        wide.to_csv(wide_csv, index=False)
        log(f"[output] Wide table saved to {wide_csv} (n={len(wide)})")

        # Quality check
        n_dup = wide["eid"].duplicated().sum()
        if n_dup > 0:
            log(f"Warning: wide table has {n_dup} duplicate eids")
        else:
            log(f"OK: all eids unique (n={wide['eid'].nunique()})")
        for col in ["gap_resid_brain", "gap_resid_huizhi", "gap_resid_baizhi"]:
            if col in wide.columns:
                n_valid = wide[col].notna().sum()
                log(f"  {col}: {n_valid} valid, {len(wide)-n_valid} missing")


# =============================================================================
# Main
# =============================================================================

def main():
    ensure_dir(OUTPUT_DIR)
    log(f"Mode: {MODE}")
    log(f"Output directory: {OUTPUT_DIR}")

    # ----- Step 1: Data preparation -----
    train_dfs, test_dfs, diseased_dfs = {}, {}, {}

    # Load disease label table (shared across modalities)
    log(f"Reading disease file: {DISEASE_FILE}")
    disease_df = pd.read_csv(DISEASE_FILE)
    for col in ["X", "Unnamed: 0"]:
        if col in disease_df.columns:
            disease_df = disease_df.drop(columns=[col])
    disease_df = disease_df[["eid", "disease"]].drop_duplicates("eid")

    for modality in MODALITIES:
        if EXISTING_SPLIT_CSVS and modality in EXISTING_SPLIT_CSVS:
            # Use pre-existing R splits
            log(f"[{modality}] Using existing train/test split")
            train_df, test_df = load_existing_split(modality, EXISTING_SPLIT_CSVS[modality])

            # Reconstruct diseased subset from cleaned + disease labels
            cleaned = pd.read_csv(CLEANED_FILES[modality])
            if "X" in cleaned.columns:
                cleaned = cleaned.drop(columns=["X"])
            cleaned = cleaned.rename(columns={"p31": "sex", "p21022": "age"})
            cleaned["sex"] = pd.to_numeric(
                cleaned["sex"].replace({"Female": 0, "Male": 1}), errors="coerce"
            )
            cleaned = cleaned.dropna().reset_index(drop=True)
            merged = cleaned.merge(disease_df, on="eid", how="left")
            diseased = merged[merged["disease"] == 1].drop(columns=["disease"]).reset_index(drop=True)
            log(f"[{modality}] Diseased n={len(diseased)}")
        else:
            # Create new splits
            train_df, test_df, diseased = preprocess_one_modality(
                modality, CLEANED_FILES[modality], disease_df,
                OUTPUT_DIR, seed=RANDOM_SEED
            )

        train_dfs[modality] = train_df
        test_dfs[modality] = test_df
        diseased_dfs[modality] = diseased

    # ----- Step 2: Training -----
    if MODE in ("train", "both"):
        log("=" * 60)
        log("Training 6 models (3 modalities x 2 sexes)")
        log("=" * 60)
        t0 = time.time()
        run_train_tasks(train_dfs, OUTPUT_DIR)
        log(f"Training complete. Total time: {time.time()-t0:.1f}s")

    # ----- Step 3: Prediction -----
    if MODE in ("predict", "both"):
        log("=" * 60)
        log("Predicting (healthy test + diseased)")
        log("=" * 60)
        t0 = time.time()
        run_predict_tasks(test_dfs, diseased_dfs, OUTPUT_DIR, train_dfs=train_dfs)
        log(f"Prediction complete. Total time: {time.time()-t0:.1f}s")

    log("Pipeline finished.")


if __name__ == "__main__":
    main()